# ImageNet: A Large-Scale Hierarchical Image Database (Deng et al., CVPR 2009)
### Triển Khai Từ Con Số 0 (From Scratch) Bằng Python & NumPy

> **Tác giả bài báo:** Jia Deng, Wei Dong, Richard Socher, Li-Jia Li, Kai Li, Li Fei-Fei  
> **Hội nghị:** CVPR 2009 (Miami, Florida)  
> **Ý nghĩa lịch sử:** Bước chuyển dịch vĩ đại mở ra kỷ nguyên **Data-Centric AI** và bệ phóng khai sinh Deep Learning hiện đại.

---

## 🎯 Mục Tiêu Bài Thực Hành
Trong notebook này, bạn sẽ tự tay khám phá và thực thi từ con số 0 các giải thuật và cấu trúc nền tảng của ImageNet:
1. **Cây phả hệ WordNet (Synset Tree):** Tổ chức các khái niệm thị giác theo quan hệ ngữ nghĩa Hypernym - Hyponym.
2. **Tổ tiên chung gần nhất (LCA) & Khoảng cách nhảy cây (Tree-Hop Distance):** Đo lường sai số nhận thức giữa các loài/vật thể.
3. **Kiểm định đám đông Amazon Mechanical Turk (AMT):** Thuật toán dừng thích ứng Bayes giúp lọc nhãn đạt độ sạch >98% với chi phí tối thiểu.
4. **Đánh giá Top-1 / Top-5 & Phân cấp:** Thay thế hàm mất mát phẳng 0/1 truyền thống bằng độ đo ngữ nghĩa.
5. **Academic Value Benchmark:** Đối chiếu định lượng sự khác biệt bản chất giữa Kỷ nguyên Tiền-ImageNet (Caltech/PASCAL) và ImageNet.

In [ ]:
# Khởi tạo thư viện toán học cơ bản
import numpy as np
print(f"[✓] Đã nạp thành công NumPy phiên bản: {np.__version__}")

## PHẦN 1: CẤU TRÚC DỮ LIỆU CÂY PHẢ HỆ WORDNET (SYNSET HIERARCHY)

Khác với danh sách phẳng (Flat list) của Caltech hay CIFAR, ImageNet neo các khái niệm hình ảnh vào cấu trúc phả hệ **WordNet**:
- **Synset (Synonym Set):** Tập từ đồng nghĩa đại diện cho 1 khái niệm duy nhất (khắc phục hiện tượng đa nghĩa).
- **Quan hệ Is-A (Cha - Con):** Cho phép máy tính hiểu rằng *Husky* là một loài *Chó*, *Chó* là *Động vật ăn thịt*, và *Động vật* là *Sinh vật sống*.

```text
                                 [entity]
                                    │
                    ┌───────────────┴───────────────┐
             [living_thing]                    [artifact]
                    │                               │
                [animal]                        [vehicle]
                    │                       ┌───────┴───────┐
                [mammal]             [wheeled_vehicle] [aircraft] ...
                    │
               [carnivore]
             ┌──────┴──────┐
         [canine]       [feline]
            │              │
          [dog]    [cat] [tiger]
       ┌────┴────┐
    [husky]  [retriever] ...
```

In [ ]:
class SynsetNode:
    """
    Đại diện cho một nút khái niệm (Synset) trong cây phả hệ WordNet.
    """
    def __init__(self, synset_id, name, parent=None):
        self.synset_id = synset_id
        self.name = name
        self.parent = parent
        self.children = []
        self.depth = 0 if parent is None else parent.depth + 1
        if parent is not None:
            parent.children.append(self)

    def is_leaf(self):
        return len(self.children) == 0

    def get_ancestors(self):
        """Trả về danh sách tất cả các nút tổ tiên từ nút hiện tại ngược về Root."""
        ancestors = []
        curr = self
        while curr is not None:
            ancestors.append(curr)
            curr = curr.parent
        return ancestors

    def get_path_to_root(self):
        ancestors = self.get_ancestors()
        ancestors.reverse()
        return " -> ".join([node.name for node in ancestors])


class WordNetHierarchy:
    """
    Quản lý toàn bộ cấu trúc đồ thị cây phả hệ WordNet.
    """
    def __init__(self, root_name="entity"):
        self.nodes = {}
        self.root = SynsetNode("n00000001", root_name)
        self.nodes["n00000001"] = self.root
        self.leaf_classes = []

    def add_synset(self, synset_id, name, parent_id):
        if parent_id not in self.nodes:
            raise ValueError(f"Không tìm thấy nút cha: {parent_id}")
        parent = self.nodes[parent_id]
        node = SynsetNode(synset_id, name, parent)
        self.nodes[synset_id] = node
        return node

    def build_sample_tree(self):
        """Xây dựng cây phả hệ mẫu gồm 12 lớp lá theo 2 nhánh Sinh vật & Đồ vật."""
        # Nhánh 1: Living Thing
        self.add_synset("n00001", "living_thing", "n00000001")
        self.add_synset("n00002", "animal", "n00001")
        self.add_synset("n00003", "mammal", "n00002")
        self.add_synset("n00004", "carnivore", "n00003")
        
        # Họ Chó
        self.add_synset("n00005", "canine", "n00004")
        self.add_synset("n00006", "dog", "n00005")
        self.add_synset("c01", "husky", "n00006")
        self.add_synset("c02", "golden_retriever", "n00006")
        self.add_synset("c03", "german_shepherd", "n00006")
        self.add_synset("c04", "wolf", "n00005")

        # Họ Mèo
        self.add_synset("n00007", "feline", "n00004")
        self.add_synset("c05", "domestic_cat", "n00007")
        self.add_synset("c06", "tiger", "n00007")

        # Nhánh 2: Artifact
        self.add_synset("n00008", "artifact", "n00000001")
        self.add_synset("n00009", "vehicle", "n00008")
        self.add_synset("n00010", "wheeled_vehicle", "n00009")
        self.add_synset("c07", "car", "n00010")
        self.add_synset("c08", "truck", "n00010")
        self.add_synset("c09", "bus", "n00010")
        self.add_synset("c10", "motorcycle", "n00010")

        self.add_synset("n00011", "aircraft", "n00009")
        self.add_synset("c11", "airplane", "n00011")
        self.add_synset("n00012", "watercraft", "n00009")
        self.add_synset("c12", "speedboat", "n00012")

        self.leaf_classes = [node for node in self.nodes.values() if node.is_leaf()]
        self.leaf_classes.sort(key=lambda x: x.synset_id)

    def find_lowest_common_ancestor(self, node_a, node_b):
        """Tìm Tổ tiên chung gần nhất (LCA) của 2 nút trong cây."""
        ancestors_a = set(node_a.get_ancestors())
        curr = node_b
        while curr is not None:
            if curr in ancestors_a:
                return curr
            curr = curr.parent
        return self.root

    def compute_tree_distance(self, node_a, node_b):
        """
        Tính khoảng cách nhảy cây (Tree-Hop Distance):
        D_tree(A, B) = Depth(A) + Depth(B) - 2 * Depth(LCA(A, B))
        """
        lca = self.find_lowest_common_ancestor(node_a, node_b)
        return node_a.depth + node_b.depth - 2 * lca.depth

    def get_distance_matrix(self):
        """Tạo ma trận khoảng cách nhảy cây kích thước (C, C)."""
        C = len(self.leaf_classes)
        dist_matrix = np.zeros((C, C), dtype=np.int32)
        for i in range(C):
            for j in range(C):
                dist_matrix[i, j] = self.compute_tree_distance(self.leaf_classes[i], self.leaf_classes[j])
        return dist_matrix

    def render_ascii_tree(self, node=None, prefix="", is_last=True):
        """Hiển thị cây phả hệ WordNet dưới dạng sơ đồ ASCII."""
        if node is None:
            node = self.root
        connector = "└── " if is_last else "├── "
        leaf_tag = " [LEAF]" if node.is_leaf() else ""
        print(prefix + connector + f"{node.name} ({node.synset_id}){leaf_tag}")
        prefix += "    " if is_last else "│   "
        child_count = len(node.children)
        for i, child in enumerate(node.children):
            self.render_ascii_tree(child, prefix, i == (child_count - 1))

print("[✓] Đã định nghĩa xong SynsetNode và WordNetHierarchy.")

In [ ]:
# Khởi tạo và hiển thị trực quan Cây Phả Hệ WordNet
tree = WordNetHierarchy()
tree.build_sample_tree()

print(f"[*] Tổng số nút trong cây: {len(tree.nodes)}")
print(f"[*] Số lượng lớp lá (khái niệm cụ thể): {len(tree.leaf_classes)}\n")
print("--- SƠ ĐỒ CÂY PHẢ HỆ WORDNET (ASCII TREE) ---")
tree.render_ascii_tree()
print("-" * 45)

In [ ]:
# Phân tích khoảng cách nhảy cây (Tree-Hop Distance) giữa các khái niệm
node_husky = tree.nodes["c01"]
node_golden = tree.nodes["c02"]
node_wolf = tree.nodes["c04"]
node_cat = tree.nodes["c05"]
node_car = tree.nodes["c07"]
node_airplane = tree.nodes["c11"]

pairs = [
    (node_husky, node_golden, "Cùng giống chó (Gần gũi nhất)"),
    (node_husky, node_wolf, "Cùng họ Chó (Canine)"),
    (node_husky, node_cat, "Cùng bộ Ăn thịt (Carnivore)"),
    (node_husky, node_car, "Chó vs Ô tô (Chỉ chung nút Entity gốc)"),
    (node_car, node_airplane, "Cùng nhánh Phương tiện (Vehicle)")
]

print(f"{'Cặp Khái Niệm':<32} | {'Tổ Tiên Chung (LCA)':<18} | {'Khoảng Cách Tree-Hop':<20} | Ghi Chú")
print("-" * 90)
for a, b, note in pairs:
    lca = tree.find_lowest_common_ancestor(a, b)
    dist = tree.compute_tree_distance(a, b)
    pair_str = f"{a.name} <-> {b.name}"
    print(f"{pair_str:<32} | {lca.name:<18} | {dist:<20} | {note}")

## PHẦN 2: THUẬT TOÁN KIỂM ĐỊNH ĐÁM ĐÔNG AMAZON MECHANICAL TURK (AMT)

Để thu thập 14 triệu bức ảnh sạch, nhóm ImageNet không gán nhãn thủ công từng ảnh mà tận dụng nền tảng vi mô Amazon Mechanical Turk (AMT).

**Thách thức:** Thợ trên AMT có độ chính xác dao động từ người tận tâm (95%) đến người bấm bừa (55%).  
**Giải pháp của ImageNet:** Dùng **Cập nhật Bayes theo tỷ số hàm hợp lý (Log-Likelihood Ratio)** và thuật toán **Dừng thích ứng (Adaptive Stopping)**:
$$\text{Log-Odds}_t = \text{Log-Odds}_{t-1} + \ln \frac{P(\text{Vote} \mid \text{Concept})}{P(\text{Vote} \mid \text{No Concept})}$$
$$\text{Posterior} = \frac{1}{1 + e^{-\text{Log-Odds}}}$$

In [ ]:
class AMTQualityVerifier:
    """
    Mô phỏng thuật toán kiểm định chất lượng nhãn đám đông của ImageNet.
    """
    def __init__(self, confidence_threshold=0.98, prior_prob=0.25):
        self.confidence_threshold = confidence_threshold
        self.prior_prob = prior_prob

    def simulate_crowd_verification(self, true_has_concept, worker_accuracies):
        log_odds = np.log(self.prior_prob / (1.0 - self.prior_prob))
        votes_collected = 0

        for acc in worker_accuracies:
            votes_collected += 1
            if true_has_concept:
                vote = np.random.rand() < acc
            else:
                vote = np.random.rand() > acc

            if vote:
                llr = np.log(acc / (1.0 - acc + 1e-12))
            else:
                llr = np.log((1.0 - acc) / (acc + 1e-12))

            log_odds += llr
            posterior = 1.0 / (1.0 + np.exp(-log_odds))

            # Điều kiện dừng thích ứng: vượt ngưỡng tin cậy thì dừng ngay lập tức
            if posterior >= self.confidence_threshold or posterior <= (1.0 - self.confidence_threshold):
                break

        accepted = posterior >= self.confidence_threshold
        return accepted, votes_collected, posterior

print("[✓] Đã định nghĩa AMTQualityVerifier.")

In [ ]:
# Chạy mô phỏng kiểm định trên 1,000 bức ảnh ứng viên
verifier = AMTQualityVerifier(confidence_threshold=0.98, prior_prob=0.25)

np.random.seed(42)
worker_pool = [0.90, 0.85, 0.55, 0.92, 0.88, 0.60, 0.95, 0.82]

n_test_images = 1000
actual_positives = 300
accepted_correctly = 0
rejected_correctly = 0
total_votes = 0

for i in range(n_test_images):
    has_concept = (i < actual_positives)
    np.random.shuffle(worker_pool)
    accepted, votes, prob = verifier.simulate_crowd_verification(has_concept, worker_pool)
    total_votes += votes

    if has_concept and accepted:
        accepted_correctly += 1
    elif not has_concept and not accepted:
        rejected_correctly += 1

precision = accepted_correctly / (accepted_correctly + (n_test_images - actual_positives - rejected_correctly) + 1e-9)
recall = accepted_correctly / actual_positives
avg_votes = total_votes / n_test_images

print(f"[*] Kết quả kiểm định trên {n_test_images} bức ảnh ứng viên từ Web:")
print(f"    - Độ chuẩn xác nhãn chấp nhận (Precision): {precision * 100:.2f}% (Rất sạch!)")
print(f"    - Tỷ lệ giữ lại ảnh chuẩn (Recall):        {recall * 100:.2f}%")
print(f"    - Số phiếu trung bình mỗi ảnh:             {avg_votes:.1f} lượt thợ/ảnh (Tiết kiệm ngân sách!)")

## PHẦN 3: ĐỘ ĐO ĐÁNH GIÁ PHÂN CẤP (TOP-1, TOP-5 & TREE-HOP ERROR)

- **Top-1 Accuracy:** Nhãn có điểm số cao nhất phải chính xác tuyệt đối.
- **Top-5 Accuracy:** Nhãn thực tế nằm trong top 5 dự đoán có điểm số cao nhất của mô hình.
- **Mean Tree-Hop Distance:** Đo chi phí lỗi dựa trên khoảng cách cây phả hệ ngữ nghĩa.

In [ ]:
def compute_top_k_accuracy(predictions, targets, k=5):
    k = min(k, predictions.shape[1])
    top_k_preds = np.argsort(predictions, axis=1)[:, -k:]
    correct = np.any(top_k_preds == targets[:, None], axis=1)
    return np.mean(correct)

def compute_hierarchical_cost(predictions, targets, distance_matrix):
    top_1_preds = np.argmax(predictions, axis=1)
    costs = distance_matrix[targets, top_1_preds]
    return np.mean(costs)

print("[✓] Đã định nghĩa compute_top_k_accuracy và compute_hierarchical_cost.")

## PHẦN 4: ĐỐI CHIẾU & ĐÁNH GIÁ GIÁ TRỊ HỌC THUẬT (ACADEMIC VALUE BENCHMARK)

Chúng ta sẽ so sánh 2 mô hình:
- **Mô hình A (Tiền-ImageNet):** Phân loại phẳng không cấu trúc, khi đoán sai thì phân bố sai số hỗn loạn trên toàn bộ các lớp (lỗi xuyên miền nghiêm trọng).
- **Mô hình B (Tư duy Phân cấp ImageNet):** Học được đặc trưng cấu trúc ngữ nghĩa, khi sai chỉ nhầm lẫn giữa các lớp có quan hệ họ hàng gần gũi.

In [ ]:
from imagenet_scratch import run_academic_value_comparison

# Thực thi đối chiếu giá trị học thuật
run_academic_value_comparison()

## 🎓 TỔNG KẾT & BÀI HỌC CỐT LÕI TỪ BÀI BÁO IMAGENET (CVPR 2009)

1. **Chuyển dịch sang Data-Centric AI:** Thuật toán thông minh không thể bù đắp cho dữ liệu nghèo nàn. ImageNet chứng minh dữ liệu khổng lồ và đa dạng mới là động cơ chính để thuật toán học được biểu diễn thị giác tổng quát.
2. **Cấu trúc Cây Phả Hệ WordNet:** Giúp bài toán thị giác máy tính phản ánh đúng nhận thức của con người. Đo lường lỗi bằng Tree-Hop Distance phản ánh sự thông minh của mô hình công bằng hơn lỗi 0/1 phẳng.
3. **Kiểm định đám đông thích ứng Bayes:** Cung cấp phương pháp luận chuẩn xác để làm sạch dữ liệu lớn từ internet với chi phí khả thi.
4. **Bệ phóng AlexNet 2012:** Không có ImageNet, sẽ không có cuộc cách mạng Deep Learning hiện đại bùng nổ vào năm 2012.